# Training IBM Model 1 on StackOverflow train data

This notebook trains the lexical translation model on the StackOverflow training collection using the lemmatized `text` field.

If you want the original non-lemmatized tokens instead, change `FIELD_NAME` to `text_unlemm`.

### What this notebook does
1. Read `QuestionFields.jsonl` and `AnswerFields.jsonl` from `demo/collections/stackoverflow_train/input_data/train`.
2. Extract the chosen text field into `derived_data/bitext/question_<field>` and `derived_data/bitext/answer_<field>`.
3. Run `./giza/create_tran.sh` to train IBM Model 1.
4. Run `./giza/filter_tran_table_and_voc.sh` to prune and binarize the translation tables.

In [8]:
# running on tran !

In [9]:
import json
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path('/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP')
SCRIPTS_DIR = REPO_ROOT / 'demo' / 'flexneuart_scripts'
COLLECT_ROOT = (REPO_ROOT / 'demo' / 'collections').resolve()
COLLECT_NAME = 'stackoverflow_tran'
INPUT_SPLIT = 'tran'
FIELD_NAME = 'text'
BITEXT_SUBDIR = 'bitext'
MODEL1_SUBDIR = 'giza'

os.environ['COLLECT_ROOT'] = str(COLLECT_ROOT)
collect_dir = COLLECT_ROOT / COLLECT_NAME
input_split_dir = collect_dir / 'input_data' / INPUT_SPLIT
bitext_dir = collect_dir / 'derived_data' / BITEXT_SUBDIR
bitext_dir.mkdir(parents=True, exist_ok=True)

q_jsonl = input_split_dir / 'QuestionFields.jsonl'
a_jsonl = input_split_dir / 'AnswerFields.jsonl'
q_bitext = bitext_dir / f'question_{FIELD_NAME}'
a_bitext = bitext_dir / f'answer_{FIELD_NAME}'

print('REPO_ROOT    =', REPO_ROOT)
print('SCRIPTS_DIR  =', SCRIPTS_DIR)
print('COLLECT_ROOT =', COLLECT_ROOT)
print('COLLECT_NAME =', COLLECT_NAME)
print('INPUT_SPLIT  =', INPUT_SPLIT)
print('FIELD_NAME   =', FIELD_NAME)
print('BITEXT_DIR   =', bitext_dir)

for path in [q_jsonl, a_jsonl]:
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')

REPO_ROOT    = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP
SCRIPTS_DIR  = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/flexneuart_scripts
COLLECT_ROOT = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
COLLECT_NAME = stackoverflow_tran
INPUT_SPLIT  = tran
FIELD_NAME   = text
BITEXT_DIR   = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/bitext


In [10]:
def export_field_lines(src_jsonl: Path, dst_txt: Path, field_name: str) -> int:
    qty = 0
    with src_jsonl.open('r', encoding='utf-8', errors='ignore') as fi, dst_txt.open('w', encoding='utf-8') as fo:
        for line in fi:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            text = obj.get(field_name, '')
            fo.write(text.replace('\n', ' ').strip() + '\n')
            qty += 1
    return qty

q_qty = export_field_lines(q_jsonl, q_bitext, FIELD_NAME)
a_qty = export_field_lines(a_jsonl, a_bitext, FIELD_NAME)

print('Wrote bitext question file:', q_bitext)
print('Wrote bitext answer file:  ', a_bitext)
print('Questions:', q_qty)
print('Answers:  ', a_qty)
if q_qty != a_qty:
    raise RuntimeError(f'Question/answer line count mismatch: {q_qty} vs {a_qty}')

Wrote bitext question file: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/bitext/question_text
Wrote bitext answer file:   /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/bitext/answer_text
Questions: 5772614
Answers:   5772614


In [11]:
cmd = ['bash', './giza/create_tran.sh', COLLECT_NAME, FIELD_NAME]
print('Running in', SCRIPTS_DIR)
print('Command:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'IBM Model 1 training failed with code {res.returncode}')

Running in /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/flexneuart_scripts
Command: bash ./giza/create_tran.sh stackoverflow_tran text
Using collection root: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
GIZA (output) sub-directory:  giza
Bitext sub-directory:         bitext
Source dir prefix:            /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/bitext
Target dir prefix:            /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/giza
Sample probability:           1
Creating: 
Dir=/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/giza/text.orig
Creating: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/giz

In [12]:
perp_path = collect_dir / 'derived_data' / MODEL1_SUBDIR / f'{FIELD_NAME}.orig' / 'output.perp'
print('Perplexity file:', perp_path)
if not perp_path.exists():
    raise FileNotFoundError(f'Missing perplexity file: {perp_path}')
print(perp_path.read_text(encoding='utf-8', errors='ignore'))

Perplexity file: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/giza/text.orig/output.perp
#trnsz	tstsz	iter	model	trn-pp		test-pp		trn-vit-pp		tst-vit-pp
10278282	0	0	Model1	3.38255e+06		N/A		inf		N/A
10278282	0	1	Model1	1555.45		N/A		inf		N/A
10278282	0	2	Model1	1197.11		N/A		inf		N/A
10278282	0	3	Model1	1112.85		N/A		inf		N/A
10278282	0	4	Model1	1083.96		N/A		inf		N/A



In [13]:
min_tran_prob = 2.5e-3 # as said in paper
top_word_qty = 100000 # not sure if it is this
cmd = [
    'bash', './giza/filter_tran_table_and_voc.sh',
    COLLECT_NAME, FIELD_NAME, str(min_tran_prob), str(top_word_qty)
]
print('Running in', SCRIPTS_DIR)
print('Command:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'Filtering translation table failed with code {res.returncode}')

Running in /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/flexneuart_scripts
Command: bash ./giza/filter_tran_table_and_voc.sh stackoverflow_tran text 0.0025 100000
Using collection root: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
 This script uses (but doesn't modify), e.g. the data created by ./giza/create_tran.sh which is placed in directory:
/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/giza/text.orig
The filtered output is stored in the following directory:
/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/giza/text
Filtering vocabularies : '/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/giza/text.orig' -> '/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-I

### Notes
- `text` is the lemmatized field in this StackOverflow collection, so this notebook trains IBM Model 1 on lemmas.
- To train on original words instead, change `FIELD_NAME` to `text_unlemm` and rerun from the bitext export cell onward.
- The trained model is written under `demo/collections/stackoverflow_train/derived_data/giza/text.orig` and the filtered version under `demo/collections/stackoverflow_train/derived_data/giza/text`.